In [0]:

data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("HealthcarePipeline").getOrCreate()

In [0]:
# 1. Create DataFrame
df = spark.createDataFrame(data, columns)
df.show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
# 2. Add required derived columns
df2 = df.withColumn(
    "total_bill",
    col("consultation_fee") + (col("tests_count") * 500)
).withColumn(
    "patient_type",
    when(col("total_bill") >= 6000, "High")
    .when(col("total_bill") >= 4000, "Medium")
    .otherwise("Low")
)
df2.show()

+--------+------------+---------+-----------+----------------+-----------+----------+------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_bill|patient_type|
+--------+------------+---------+-----------+----------------+-----------+----------+------------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|      Medium|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|      Medium|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|         Low|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|        High|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7500|        High|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|      Medium|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      5500|      Medium|
|     108|

In [0]:
# 3. Filter high-value patients
high_value_df = df2.filter(col("patient_type") == "High")
high_value_df.show()

+--------+------------+---------+----------+----------------+-----------+----------+------------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|total_bill|patient_type|
+--------+------------+---------+----------+----------------+-----------+----------+------------+
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|      6000|        High|
|     105|Vikram Singh|  Chennai| Neurology|            7000|          1|      7500|        High|
+--------+------------+---------+----------+----------------+-----------+----------+------------+



In [0]:
# 4. Perform aggregation by department
agg_df = df2.groupBy("department").agg(
    count("*").alias("total_patients"),
    sum("total_bill").alias("total_revenue"),
    avg("total_bill").alias("avg_bill")
)
agg_df.show()

+-----------+--------------+-------------+-----------------+
| department|total_patients|total_revenue|         avg_bill|
+-----------+--------------+-------------+-----------------+
| Cardiology|             3|        17000|5666.666666666667|
|Orthopedics|             2|         8500|           4250.0|
|Dermatology|             2|         4500|           2250.0|
|  Neurology|             1|         7500|           7500.0|
+-----------+--------------+-------------+-----------------+



In [0]:
# 5. Sort results appropriately
sorted_df = agg_df.orderBy(col("total_revenue").desc())
sorted_df.show()

+-----------+--------------+-------------+-----------------+
| department|total_patients|total_revenue|         avg_bill|
+-----------+--------------+-------------+-----------------+
| Cardiology|             3|        17000|5666.666666666667|
|Orthopedics|             2|         8500|           4250.0|
|  Neurology|             1|         7500|           7500.0|
|Dermatology|             2|         4500|           2250.0|
+-----------+--------------+-------------+-----------------+



In [0]:
# 1. Convert DataFrame to Temp View
df2.createOrReplaceTempView("patient_visits")

In [0]:
%sql
-- 2A. Fetch specific department records
SELECT * FROM patient_visits WHERE department = 'Cardiology';

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Medium
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Medium


In [0]:
%sql
-- 2B. Calculate revenue per city
SELECT 
    city,
    SUM(total_bill) AS total_revenue
FROM patient_visits
GROUP BY city
ORDER BY total_revenue DESC;

city,total_revenue
Bangalore,8500
Chennai,7500
Hyderabad,5500
Ahmedabad,5500
Kolkata,4500
Delhi,4000
Mumbai,2000


In [0]:
%sql
-- 2C. Identify top patients
SELECT *
FROM patient_visits
ORDER BY total_bill DESC
LIMIT 5;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_type
105,Vikram Singh,Chennai,Neurology,7000,1,7500,High
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Medium
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Medium
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Medium


In [0]:
%sql
-- 2D. Count patients per department
SELECT 
    department,
    COUNT(*) AS patient_count
FROM patient_visits
GROUP BY department
ORDER BY patient_count DESC;

department,patient_count
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


In [0]:
%sql
-- 1. Create a Delta Table from dataset
CREATE TABLE patient_delta
USING DELTA
AS
SELECT * FROM patient_visits;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- 2. Insert new records
INSERT INTO patient_delta VALUES
(109,"Rohit Mehta","Mumbai","Cardiology",5000,2,6000,"High"),
(110,"Divya Menon","Chennai","Neurology",7000,1,7500,"High");

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
-- 3. Update existing records
UPDATE patient_delta
SET consultation_fee = consultation_fee + 500
WHERE department = 'Cardiology';

num_affected_rows
4


In [0]:
%sql
-- 4. Delete specific records
DELETE FROM patient_delta
WHERE patient_type = 'Low';

num_affected_rows
2


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW updates AS
SELECT * FROM VALUES
(101,"Arjun Reddy","Hyderabad","Cardiology",5500,2,6500,"High"),
(111,"Neha Sharma","Delhi","Dermatology",2000,1,2500,"Low")
AS updates(
visit_id,
patient_name,
city,
department,
consultation_fee,
tests_count,
total_bill,
patient_type
);

MERGE INTO patient_delta AS target
USING updates AS source
ON target.visit_id = source.visit_id

WHEN MATCHED THEN
UPDATE SET
target.patient_name = source.patient_name,
target.city = source.city,
target.department = source.department,
target.consultation_fee = source.consultation_fee,
target.tests_count = source.tests_count,
target.total_bill = source.total_bill,
target.patient_type = source.patient_type

WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
-- 1. Retrieve table history
DESCRIBE HISTORY patient_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-05-04T07:40:25.000Z,141454688300890,azuser5812_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(9532177704483),0185a9db-a280-45d6-9581-62c4f8b7b745,0504-062849-d4ydml85-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7314, p25FileSize -> 2849, numDeletionVectorsRemoved -> 1, minFileSize -> 2849, numAddedFiles -> 1, maxFileSize -> 2849, p75FileSize -> 2849, p50FileSize -> 2849, numAddedBytes -> 2849)",null,Databricks-Runtime/18.1.x-photon-scala2.13
6,2026-05-04T07:40:24.000Z,141454688300890,azuser5812_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#16157L = cast(visit_id#16141 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(9532177704483),0185a9db-a280-45d6-9581-62c4f8b7b745,0504-062849-d4ydml85-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4521, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3664, materializeSourceTimeMs -> 418, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1455, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1747)",null,Databricks-Runtime/18.1.x-photon-scala2.13
5,2026-05-04T07:40:14.000Z,141454688300890,azuser5812_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(9532177704483),972c766a-8c73-422f-9943-171734987250,0504-062849-d4ydml85-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2864, p25FileSize -> 2793, numDeletionVectorsRemoved -> 1, minFileSize -> 2793, numAddedFiles -> 1, maxFileSize -> 2793, p75FileSize -> 2793, p50FileSize -> 2793, numAddedBytes -> 2793)",null,Databricks-Runtime/18.1.x-photon-scala2.13
4,2026-05-04T07:40:12.000Z,141454688300890,azuser5812_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(patient_type#15783 = Low)""])",null,List(9532177704483),972c766a-8c73-422f-9943-171734987250,0504-062849-d4ydml85-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1338, numDeletionVectorsUpdated -> 0, numDeletedRows -> 2, scanTimeMs -> 998, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 339)",null,Databricks-Runtime/18.1.x-photon-scala2.13
3,2026-05-04T07:40:06.000Z,141454688300890,azuser5812_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(9532177704483),59fcebcb-fcf5-43ac-b6e1-7899d9a09dcc,0504-062849-d4ydml85-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7794, p25FileSize -> 2864, numDeletionVectorsRemoved -> 2, minFileSize -> 2864, numAddedFiles -> 1, maxFileSize -> 2864, p75FileSize -> 2864, p50FileSize -> 2864, numAddedBytes -> 2864)",null,Databricks-Runtime/18.1.x-photon-scala2.13
2,2026-05-04T07:40:05.000Z,141454688300890,azuser5812_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(department#15173 = Cardiology)""])",null,List(9532177704483),59fcebcb-fcf5-43ac-b6e1-7899d9a0

In [0]:
%sql
-- 2. Query an older version of the table
SELECT *
FROM patient_delta VERSION AS OF 0;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Medium
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,4000,Medium
103,Rahul Sharma,Mumbai,Dermatology,1500,1,2000,Low
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High
105,Vikram Singh,Chennai,Neurology,7000,1,7500,High
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Medium
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Medium
108,Meera Iyer,Bangalore,Dermatology,1500,2,2500,Low


In [0]:
%sql
-- 3. Explain the effect of VACUUM

-- VACUUM removes old files (older versions)
-- Helps in saving storage space
-- After VACUUM → Time Travel to old versions may NOT be possible
-- Default retention = 7 days

In [0]:
%sql
-- 4. Execute VACUUM (Dry Run)
-- Shows which files will be deleted (no actual deletion)
VACUUM patient_delta RETAIN 168 HOURS DRY RUN;
-- 168 hours = 7 days

path


In [0]:
# 1. Save dataset as Parquet
df2.write.mode("overwrite").saveAsTable("patient_parquet")

In [0]:
# 2. Convert Parquet dataset to Delta
spark.sql("""
CREATE OR REPLACE TABLE patient_delta
USING DELTA
AS SELECT * FROM patient_parquet
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# 3. Validate conversion
spark.sql("SELECT * FROM patient_delta").show()
spark.sql("DESCRIBE DETAIL patient_delta").show()

+--------+------------+---------+-----------+----------------+-----------+----------+------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_bill|patient_type|
+--------+------------+---------+-----------+----------------+-----------+----------+------------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|      Medium|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|      Medium|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|         Low|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|        High|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7500|        High|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|      Medium|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      5500|      Medium|
|     108|

In [0]:
%sql
-- 1. Create target Delta table
CREATE TABLE patient_target
USING DELTA
AS
SELECT * FROM patient_visits;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- 2. Create dataset (daily updates)
CREATE OR REPLACE TEMP VIEW daily_updates AS
SELECT * FROM VALUES
(101,"Arjun Reddy","Hyderabad","Cardiology",6000,2,7000,"High"), 
(112,"Pooja Shah","Mumbai","Neurology",8000,1,8500,"High") 
AS daily_updates(
visit_id,
patient_name,
city,
department,
consultation_fee,
tests_count,
total_bill,
patient_type
);

In [0]:
%sql
-- 3 Implement Incremental Load logic
MERGE INTO patient_target AS target
USING daily_updates AS source
ON target.visit_id = source.visit_id

WHEN MATCHED THEN
UPDATE SET
target.patient_name = source.patient_name,
target.city = source.city,
target.department = source.department,
target.consultation_fee = source.consultation_fee,
target.tests_count = source.tests_count,
target.total_bill = source.total_bill,
target.patient_type = source.patient_type

WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
spark.sql("USE CATALOG my_catelog")

DataFrame[]

In [0]:
spark.sql("USE SCHEMA my_schema")

DataFrame[]

In [0]:
spark.sql("""
          CREATE TABLE students(
              id INT,
              name STRING,
              age INT
          )
          """)

DataFrame[]

In [0]:
spark.sql("""
          INSERT INTO students VALUES
          (1, 'John', 20),
          (2, 'Alice', 22),
          (3, 'Bob', 21)
          """)


DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("SELECT * FROM students").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1| John| 20|
|  2|Alice| 22|
|  3|  Bob| 21|
+---+-----+---+



In [0]:
spark.sql("SHOW TABLES IN my_catelog.my_schema").show()

+---------+--------------+-----------+
| database|     tableName|isTemporary|
+---------+--------------+-----------+
|my_schema|      students|      false|
|         |patient_visits|       true|
+---------+--------------+-----------+



In [0]:
spark.sql("""
CREATE OR REPLACE TABLE my_catelog.my_schema.students_adults AS
SELECT * FROM my_catelog.my_schema.students
WHERE age >= 21
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("SELECT * FROM my_catelog.my_schema.students_adults").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  2|Alice| 22|
|  3|  Bob| 21|
+---+-----+---+



In [0]:
spark.sql("""
GRANT SELECT ON TABLE my_catelog.my_schema.students
TO `azuser5812_mml.local@karthikirisoutlook.onmicrosoft.com`
""")

DataFrame[]

In [0]:
spark.sql("SHOW GRANTS ON TABLE my_catelog.my_schema.students").show()

+--------------------+----------+----------+--------------------+
|           Principal|ActionType|ObjectType|           ObjectKey|
+--------------------+----------+----------+--------------------+
|azuser5812_mml.lo...|    SELECT|     TABLE|my_catelog.my_sch...|
+--------------------+----------+----------+--------------------+

